# Lecture 3, optional notebook: the cannibalisation curve

Companion to the lecture 3 worksheet. **Nothing in the worksheet needs this
notebook**: the evidence for part 2 is printed there. This is for anyone who
wants to reproduce it, turn the knobs, and try the questions at the end.

It runs the note's hourly dispatch model (section 3.2) on the ten-plant
fleet of the note's Table 1, against DK1's 2024 load, wind and solar
profiles at every eighth hour of the year — the same instance as the note's
own `notebooks/02-intermittency.ipynb`.

In [ ]:
# Make the note's models importable, wherever you launched Jupyter from.
import sys
from pathlib import Path

here = Path.cwd()
candidates = [c for p in [here, *here.parents]
              for c in (p, p / "notes" / "energy-system-models")]
note = next(c for c in candidates if (c / "model" / "dispatch.py").exists())
sys.path.insert(0, str(note))
PROCESSED = note / "data" / "processed"

import pandas as pd
import matplotlib.pyplot as plt

from model import dispatch, dispatch_t
from pipeline.run_dispatch import CAPACITY_MW

tech = dispatch.read_tech(PROCESSED / "technology_costs_small.csv")
profiles = pd.read_csv(PROCESSED / "profiles_dk1_2024.csv", index_col="time", parse_dates=True)

STRIDE = 8
sample = profiles.iloc[::STRIDE]
load = sample["load_mw"].rename("load")
availability = pd.DataFrame({
    "solar_pv":      sample["solar_pu"],
    "wind_onshore":  sample["wind_onshore_pu"],
    "wind_offshore": sample["wind_offshore_pu"],
})
WIND, SOLAR = ["wind_onshore", "wind_offshore"], ["solar_pv"]
print(f"note root: {note}")
print(f"{len(load)} hours of DK1 2024; base capacities (MW):")
print(CAPACITY_MW.to_string())

## One solve, and the statistics of section 3.4

`statistics` returns, for a group of technologies, its share of consumption,
its capture price, the consumption-weighted base price of the note (eq. 18),
the time-average price Hirth (2013) uses instead, the value factor on each
base, curtailment, and revenue.

In [ ]:
def statistics(sol, techs, cap, co2_price=0.0):
    g, p = sol["generation"], sol["price"]
    q = g[techs].sum(axis=1)
    offered = availability[techs].mul(cap[techs], axis=1).sum(axis=1).sum()
    capture = (p * q).sum() / q.sum()
    base = dispatch_t.base_price(p, load)
    return pd.Series({
        "share": q.sum() / load.sum(),
        "capture_price": capture,
        "base_price": base,
        "avg_price": p.mean(),
        "value_factor": capture / base,
        "value_factor_hirth": capture / p.mean(),
        "curtailment": 1 - q.sum() / offered,
        "revenue_meur": (p * q).sum() / 1e6,
    })


def solve_scaled(scaled, s, co2_price=0.0):
    cap = CAPACITY_MW.copy()
    cap[scaled] *= s
    return dispatch_t.solve(tech, cap, load, availability, co2_price=co2_price), cap


sol, cap = solve_scaled(WIND, 1.0)
statistics(sol, WIND, cap).round(3)

## The three sweeps of the worksheet

Each point is a full re-solve of the year. This reproduces the worksheet's
table (`figures/make_evidence.py` writes it from the same code).

In [ ]:
SCALES = [0.1, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6]
RATIO_SCALES = [1, 2, 4, 8, 16, 24]


def sweep(scaled, measured, scales, co2_price=0.0):
    rows = []
    for s in scales:
        sol, cap = solve_scaled(scaled, s, co2_price)
        row = statistics(sol, measured, cap)
        row["scale"] = s
        if scaled != measured:
            row["scaled_share"] = statistics(sol, scaled, cap)["share"]
        rows.append(row)
    return pd.DataFrame(rows).set_index("scale")


wind = sweep(WIND, WIND, SCALES)
solar = sweep(SOLAR, SOLAR, SCALES)
ratio = sweep(SOLAR, WIND, RATIO_SCALES)
wind.round(3)

In [ ]:
solar.round(3)

In [ ]:
ratio.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
ax = axes[0]
ax.plot(100 * wind.share, wind.capture_price, "o-", color="#0072B2", label="wind capture price")
ax.plot(100 * wind.share, wind.base_price, "--", color="#0072B2", label="base price, wind sweep")
ax.plot(100 * solar.share, solar.capture_price, "s-", color="#D55E00", label="solar capture price")
ax.plot(100 * solar.share, solar.base_price, "--", color="#D55E00", label="base price, solar sweep")
ax.set_xlabel("share of consumption (%)"); ax.set_ylabel("EUR/MWh"); ax.legend(frameon=False, fontsize=8)
ax = axes[1]
ax.plot(100 * wind.share, wind.value_factor, "o-", color="#0072B2", label="wind")
ax.plot(100 * solar.share, solar.value_factor, "s-", color="#D55E00", label="solar")
ax.axhline(1, color="#999999", lw=0.8)
ax.set_xlabel("share of consumption (%)"); ax.set_ylabel("value factor"); ax.legend(frameon=False)
ax = axes[2]
ax.plot(100 * ratio.scaled_share, ratio.value_factor, "o-", color="#0072B2", label="wind value factor")
ax2 = ax.twinx()
ax2.plot(100 * ratio.scaled_share, ratio.revenue_meur, "s--", color="#D55E00", label="wind revenue (MEUR)")
ax.set_xlabel("solar share of consumption (%)"); ax.set_ylabel("wind value factor"); ax2.set_ylabel("wind revenue (MEUR)")
fig.tight_layout(); plt.show()

## Questions to explore

Each is one change to the code above. None is needed for the worksheet.

1. **The carbon price** (worksheet part 2, stretch). Hirth (2013, §5.5) finds
   the effect of a carbon price on wind's value factor ambiguous through
   three channels. Rerun the wind sweep with `co2_price=85`. Which way does
   the value factor move at $s = 2$, and which of his channels is doing it?
   Look at `dispatch.marginal_cost(tech, 85)` for the re-sorted merit order.
2. **Hirth's base price.** Compare `value_factor` with `value_factor_hirth`
   along the wind and the solar sweeps. For which technology do the two
   definitions differ more, and why (worksheet part 1, question 3)?
3. **Sampling.** Change `STRIDE` from 8 to 4, or to 1 for the full year
   (slower). Which numbers in the table move, and which conclusions?
4. **An outlet for the surplus.** The note's DK1 fleet (figure 3.5) has an
   import plant; this fleet has no wires at all, so surplus wind is curtailed.
   Add a constant-cost plant that can *absorb* energy — a negative-load
   "export" — or simply add a large, expensive dispatchable plant and see what
   the top of the price distribution does to the base price. Which direction
   does each push wind's value factor, and does it match the bias box at the
   end of section 3?
5. **Price-setting technology.** For $s = 1$ and $s = 4$, count in how many
   hours each technology is the marginal one (the price equals its marginal
   cost) and how much of wind's energy is sold at a price at or below onshore
   wind's own marginal cost. This is Hirth's figure 11 for this fleet.
6. **The counterexample of part 1's stretch, in data.** Along the solar
   sweep, wind's value factor *rises* at high solar shares. Decompose the
   change into the capture-price and base-price terms for each step of
   `RATIO_SCALES`. Where does the sign flip, and what happened to the
   price-setting plant in the midday hours there?

In [ ]:
# Try it here.